In [1]:
# numpy, pandas, scipy, scikit-learn은 Colab 기본 환경에 포함되어 있음
# LightGBM은 명시적으로 설치 (Colab에 기본 포함되어 있더라도 안전을 위해 설치 명령 명시)
%pip install -q lightgbm


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np


# 센서 데이터 (raw time-series)와 레이블 데이터를 각각 로드
# 센서 데이터의 timestamp는 ms 단위 정수로 통일하여 후속 윈도잉 처리에서 정확하게 비교
_DTYPES_SENSOR = {
    'pid': str,
    'accel_x': np.float32,
    'accel_y': np.float32,
    'accel_z': np.float32,
    'eda': np.float32,
    'heart_rate': np.float32,
    'temperature': np.float32,
}

TRAIN_DATA = pd.read_csv('train-sensor.csv', dtype=_DTYPES_SENSOR)
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA = pd.read_csv('test-sensor.csv', dtype=_DTYPES_SENSOR)
TEST_LABEL = pd.read_csv('test-label.csv', index_col='id')

# timestamp 컬럼은 정수형 ms로 캐스팅 (CSV 파싱 후 문자열 공백이 남는 경우를 대비해 strip)
for _df in (TRAIN_DATA, TEST_DATA):
    _df['timestamp'] = pd.to_numeric(_df['timestamp'], errors='coerce').astype('int64')
TRAIN_LABEL['timestamp'] = pd.to_numeric(TRAIN_LABEL['timestamp'], errors='coerce').astype('int64')
TEST_LABEL['timestamp'] = pd.to_numeric(TEST_LABEL['timestamp'], errors='coerce').astype('int64')

# 레이블 데이터의 stress 컬럼은 float -> int로 캐스팅 (train만)
TRAIN_LABEL['stress'] = TRAIN_LABEL['stress'].astype(int)

print('TRAIN_DATA :', TRAIN_DATA.shape)
print('TRAIN_LABEL:', TRAIN_LABEL.shape)
print('TEST_DATA  :', TEST_DATA.shape)
print('TEST_LABEL :', TEST_LABEL.shape)
print()
print('Train pids:', sorted(TRAIN_LABEL['pid'].unique()))
print('Test  pids:', sorted(TEST_LABEL['pid'].unique()))
print()
print('Class distribution in train:')
print(TRAIN_LABEL['stress'].value_counts().sort_index())

TRAIN_DATA : (4694400, 8)
TRAIN_LABEL: (815, 3)
TEST_DATA  : (5921280, 8)
TEST_LABEL : (1028, 3)

Train pids: ['43JW', 'C8Q6', 'DT5C', 'F1ZM', 'HDS9', 'P4DZ', 'TPQI']
Test  pids: ['01Z2', '2XO3', 'D1XP', 'NQRB', 'SE4Q', 'SNG7', 'TF0Y', 'Y21H']

Class distribution in train:
stress
0    162
1     66
2    587
Name: count, dtype: int64


## 1. 특성 추출 (3분 윈도우 통계)

In [3]:
# 센서 채널 정의
SENSOR_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']
WINDOW_MS = 3 * 60 * 1000  # 3분 = 180,000 ms

# 빠른 윈도잉을 위해 pid별로 정렬된 timestamp 배열과 값 배열을 미리 인덱싱
def index_sensor_by_pid(sensor_df):
    out = {}
    for pid, sub in sensor_df.groupby('pid', sort=False):
        sub = sub.sort_values('timestamp', kind='mergesort')
        ts = sub['timestamp'].to_numpy()
        vals = {c: sub[c].to_numpy(dtype=np.float64) for c in SENSOR_COLS}
        out[pid] = (ts, vals)
    return out

_train_sensor_idx = index_sensor_by_pid(TRAIN_DATA)
_test_sensor_idx = index_sensor_by_pid(TEST_DATA)
print('Indexed train pids :', list(_train_sensor_idx.keys()))
print('Indexed test  pids :', list(_test_sensor_idx.keys()))

Indexed train pids : ['P4DZ', 'HDS9', 'TPQI', 'DT5C', '43JW', 'F1ZM', 'C8Q6']
Indexed test  pids : ['D1XP', '01Z2', 'SE4Q', 'SNG7', '2XO3', 'Y21H', 'NQRB', 'TF0Y']


In [4]:
# 피험자 간 분포 차이를 줄이기 위해, 각 피험자의 전체 기록에서 robust baseline (median, IQR)을 계산
# 이 baseline으로 정규화한 특성을 추가하면 unseen subject에 대한 일반화에 도움이 됩니다.
def compute_subject_baselines(sensor_df):
    baselines = {}
    for pid, sub in sensor_df.groupby('pid', sort=False):
        b = {}
        for c in SENSOR_COLS:
            v = sub[c].to_numpy()
            v = v[np.isfinite(v)]
            if v.size == 0:
                b[c] = (0.0, 1.0)
            else:
                med = float(np.median(v))
                q75, q25 = np.percentile(v, [75, 25])
                iqr = float(max(q75 - q25, 1e-6))
                b[c] = (med, iqr)
        baselines[pid] = b
    return baselines

_train_baselines = compute_subject_baselines(TRAIN_DATA)
_test_baselines = compute_subject_baselines(TEST_DATA)
print('Computed per-subject baselines.')

Computed per-subject baselines.


In [5]:
from scipy import stats as _scipy_stats

# 한 윈도우(1차원 배열)에 대한 11개 통계 특성
_STAT_NAMES = ['mean', 'std', 'min', 'max', 'median', 'iqr', 'q25', 'q75', 'range', 'skew', 'kurt']

def _stat_features(arr):
    if arr.size == 0:
        return [np.nan] * 11
    a = arr[np.isfinite(arr)]
    if a.size == 0:
        return [np.nan] * 11
    q25, q50, q75 = np.percentile(a, [25, 50, 75])
    return [
        float(np.mean(a)),
        float(np.std(a)),
        float(np.min(a)),
        float(np.max(a)),
        float(q50),
        float(q75 - q25),
        float(q25),
        float(q75),
        float(np.max(a) - np.min(a)),
        float(_scipy_stats.skew(a)) if a.size > 2 else 0.0,
        float(_scipy_stats.kurtosis(a)) if a.size > 3 else 0.0,
    ]

# 각 레이블 시점에 대해 3분 윈도우의 특성을 추출
def extract_features(label_df, idx_dict, baselines):
    rows = []
    for row in label_df.itertuples(index=True):
        rid = row.Index
        pid = row.pid
        t_end = int(row.timestamp)
        t_start = t_end - WINDOW_MS
        feat = {'_id': rid, '_pid': pid, '_n_samples': 0}

        if pid not in idx_dict:
            # 센서 데이터에 해당 pid가 없는 경우 모든 특성을 NaN (LightGBM이 자체 처리)
            for c in SENSOR_COLS:
                for nm in _STAT_NAMES:
                    feat[f'{c}_{nm}'] = np.nan
                    feat[f'{c}_n_{nm}'] = np.nan
                feat[f'{c}_abs_sum_change'] = np.nan
                feat[f'{c}_mean_abs_change'] = np.nan
                feat[f'{c}_std_change'] = np.nan
                feat[f'{c}_slope'] = np.nan
            for nm in _STAT_NAMES:
                feat[f'accel_mag_{nm}'] = np.nan
            feat['accel_sma'] = np.nan
            rows.append(feat)
            continue

        ts, vals = idx_dict[pid]
        # 윈도우 [t_start, t_end] 내의 인덱스 찾기 (sorted ts에 대한 binary search)
        lo = np.searchsorted(ts, t_start, side='left')
        hi = np.searchsorted(ts, t_end, side='right')
        feat['_n_samples'] = int(hi - lo)

        for c in SENSOR_COLS:
            arr = vals[c][lo:hi]

            # (1) 원시 통계
            for nm, v in zip(_STAT_NAMES, _stat_features(arr)):
                feat[f'{c}_{nm}'] = v

            # (2) 피험자별 baseline 정규화 통계
            med, iqr = baselines.get(pid, {}).get(c, (0.0, 1.0))
            arr_n = (arr - med) / iqr if arr.size > 0 else arr
            for nm, v in zip(_STAT_NAMES, _stat_features(arr_n)):
                feat[f'{c}_n_{nm}'] = v

            # (3) Temporal change 특성 (강의자료 8.1)
            if arr.size >= 2:
                a_fin = arr[np.isfinite(arr)]
                if a_fin.size >= 2:
                    diffs = np.diff(a_fin)
                    feat[f'{c}_abs_sum_change'] = float(np.sum(np.abs(diffs)))
                    feat[f'{c}_mean_abs_change'] = float(np.mean(np.abs(diffs)))
                    feat[f'{c}_std_change'] = float(np.std(diffs))
                    feat[f'{c}_slope'] = float((a_fin[-1] - a_fin[0]) / max(a_fin.size - 1, 1))
                else:
                    feat[f'{c}_abs_sum_change'] = np.nan
                    feat[f'{c}_mean_abs_change'] = np.nan
                    feat[f'{c}_std_change'] = np.nan
                    feat[f'{c}_slope'] = np.nan
            else:
                feat[f'{c}_abs_sum_change'] = np.nan
                feat[f'{c}_mean_abs_change'] = np.nan
                feat[f'{c}_std_change'] = np.nan
                feat[f'{c}_slope'] = np.nan

        # (4) 가속도계 magnitude (활동 강도) 특성
        ax = vals['accel_x'][lo:hi]
        ay = vals['accel_y'][lo:hi]
        az = vals['accel_z'][lo:hi]
        if ax.size > 0:
            mag = np.sqrt(ax * ax + ay * ay + az * az)
            for nm, v in zip(_STAT_NAMES, _stat_features(mag)):
                feat[f'accel_mag_{nm}'] = v
            # Signal Magnitude Area: 활동 강도의 표준 지표
            feat['accel_sma'] = float(np.mean(np.abs(ax) + np.abs(ay) + np.abs(az)))
        else:
            for nm in _STAT_NAMES:
                feat[f'accel_mag_{nm}'] = np.nan
            feat['accel_sma'] = np.nan

        rows.append(feat)

    return pd.DataFrame(rows).set_index('_id')

print('Extracting train features...')
train_feats = extract_features(TRAIN_LABEL, _train_sensor_idx, _train_baselines)
print('Train features shape:', train_feats.shape)

print('Extracting test features...')
test_feats = extract_features(TEST_LABEL, _test_sensor_idx, _test_baselines)
print('Test features shape :', test_feats.shape)
print()
print('Sample of n_samples per window (train):', train_feats['_n_samples'].describe().round(1).to_dict())

Extracting train features...


/var/folders/w7/h8ptyfn56xnfdf1y_gqj5ssw0000gn/T/ipykernel_2034/1692183330.py:23: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  float(_scipy_stats.skew(a)) if a.size > 2 else 0.0,
/var/folders/w7/h8ptyfn56xnfdf1y_gqj5ssw0000gn/T/ipykernel_2034/1692183330.py:24: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  float(_scipy_stats.kurtosis(a)) if a.size > 3 else 0.0,


Train features shape: (815, 170)
Extracting test features...
Test features shape : (1028, 170)

Sample of n_samples per window (train): {'count': 815.0, 'mean': 5760.9, 'std': 0.3, 'min': 5760.0, '25%': 5761.0, '50%': 5761.0, '75%': 5761.0, 'max': 5761.0}


## 2. 모델링용 행렬 준비 및 클래스 가중치 계산

In [6]:
# 모델 학습용 입력 행렬 X와 레이블 y, 그리고 GroupKFold용 group 배열을 준비
y_train = TRAIN_LABEL.loc[train_feats.index, 'stress'].astype(int).to_numpy()
groups = train_feats['_pid'].to_numpy()

# 메타 컬럼 (_pid, _id, _n_samples)을 제외한 모든 수치 컬럼을 특성으로 사용
FEATURE_COLS = [c for c in train_feats.columns if not c.startswith('_')]
X_train = train_feats[FEATURE_COLS].to_numpy(dtype=np.float32)
X_test = test_feats[FEATURE_COLS].to_numpy(dtype=np.float32)
print('X_train:', X_train.shape, ' y_train:', y_train.shape)
print('X_test :', X_test.shape)
print('Number of features:', len(FEATURE_COLS))

# 클래스별 sample weight 계산 (sklearn의 'balanced' 공식과 동일)
NUM_CLASS = 3
_class_counts = np.bincount(y_train, minlength=NUM_CLASS).astype(float)
_class_weights = len(y_train) / (NUM_CLASS * np.maximum(_class_counts, 1.0))
sample_weights = _class_weights[y_train]
print('Class counts :', _class_counts.tolist())
print('Class weights:', np.round(_class_weights, 3).tolist())

X_train: (815, 168)  y_train: (815,)
X_test : (1028, 168)
Number of features: 168
Class counts : [162.0, 66.0, 587.0]
Class weights: [1.677, 4.116, 0.463]


## 3. Subject-Disjoint Cross-Validation + Multi-Seed LightGBM Ensemble

In [7]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import lightgbm as lgb

# LightGBM 하이퍼파라미터 - 작은 데이터셋(815)에 맞춰 정칙화 강하게 설정
LGB_PARAMS = dict(
    objective='multiclass',
    num_class=NUM_CLASS,
    metric='multi_logloss',
    learning_rate=0.03,
    num_leaves=31,
    min_data_in_leaf=8,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    lambda_l1=0.1,
    lambda_l2=0.1,
    verbosity=-1,
    n_jobs=-1,
)

# 다중 seed: 동일한 GroupKFold도 그룹 배치 순서가 seed에 따라 달라지도록 그룹 인덱스를 셔플
SEEDS = [42, 7, 2025]

_unique_groups = np.unique(groups)
N_SPLITS = min(5, len(_unique_groups))
print(f'GroupKFold: {N_SPLITS} splits over {len(_unique_groups)} subjects')

oof_proba = np.zeros((len(y_train), NUM_CLASS), dtype=np.float64)
test_proba_sum = np.zeros((len(X_test), NUM_CLASS), dtype=np.float64)
n_models = 0
fold_scores = []

for seed in SEEDS:
    print(f'\n=== Seed {seed} ===')
    rng = np.random.default_rng(seed)
    perm = rng.permutation(_unique_groups)
    perm_map = {g: i for i, g in enumerate(perm)}
    shuffled_group_idx = np.array([perm_map[g] for g in groups])

    gkf = GroupKFold(n_splits=N_SPLITS)
    for fold, (tr_i, va_i) in enumerate(gkf.split(X_train, y_train, groups=shuffled_group_idx)):
        X_tr, X_va = X_train[tr_i], X_train[va_i]
        y_tr, y_va = y_train[tr_i], y_train[va_i]
        w_tr = sample_weights[tr_i]

        # Validation에서도 클래스 균형 가중치를 사용해, multi_logloss가 BA의 매끄러운 surrogate가 되도록 함
        cnt_va = np.bincount(y_va, minlength=NUM_CLASS).astype(float)
        cw_va = len(y_va) / (NUM_CLASS * np.maximum(cnt_va, 1.0))
        w_va = cw_va[y_va]

        params = dict(LGB_PARAMS, seed=seed + fold)
        train_ds = lgb.Dataset(X_tr, label=y_tr, weight=w_tr)
        val_ds = lgb.Dataset(X_va, label=y_va, weight=w_va, reference=train_ds)

        model = lgb.train(
            params,
            train_ds,
            num_boost_round=2000,
            valid_sets=[val_ds],
            callbacks=[
                lgb.early_stopping(stopping_rounds=100, verbose=False),
                lgb.log_evaluation(period=0),
            ],
        )

        proba_va = model.predict(X_va, num_iteration=model.best_iteration)
        oof_proba[va_i] += proba_va / len(SEEDS)
        test_proba_sum += model.predict(X_test, num_iteration=model.best_iteration)
        n_models += 1

        ba_fold = balanced_accuracy_score(y_va, np.argmax(proba_va, axis=1))
        fold_scores.append(ba_fold)
        val_groups = sorted(np.unique(groups[va_i]).tolist())
        print(f'  Fold {fold}: BA={ba_fold:.4f} | best_iter={model.best_iteration} | val pids={val_groups}')

print(f'\nMean per-fold BA: {np.mean(fold_scores):.4f} (std={np.std(fold_scores):.4f})')

# Out-of-fold (subject-disjoint) Balanced Accuracy = 정직한 일반화 성능 추정
oof_pred = np.argmax(oof_proba, axis=1)
oof_ba = balanced_accuracy_score(y_train, oof_pred)
print(f'\n=== Subject-disjoint OOF Balanced Accuracy: {oof_ba:.4f} ===')
print('Per-class recall (= diagonal of normalized CM):')
_cm = confusion_matrix(y_train, oof_pred, labels=[0, 1, 2])
_per_class_recall = _cm.diagonal() / np.maximum(_cm.sum(axis=1), 1)
for i, r in enumerate(_per_class_recall):
    print(f'  class {i}: recall={r:.4f}  (n_true={_cm.sum(axis=1)[i]})')
print('\nConfusion matrix (rows=true, cols=pred):')
print(pd.DataFrame(_cm, index=['true_0', 'true_1', 'true_2'], columns=['pred_0', 'pred_1', 'pred_2']))

GroupKFold: 5 splits over 7 subjects

=== Seed 42 ===


/Users/dayana/git repo/machine-learning-class/lab3/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  Fold 0: BA=0.4648 | best_iter=24 | val pids=['C8Q6']
  Fold 1: BA=0.3333 | best_iter=1 | val pids=['P4DZ']


/Users/dayana/git repo/machine-learning-class/lab3/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  Fold 2: BA=0.4701 | best_iter=15 | val pids=['F1ZM']


...

  Fold 0: BA=0.4613 | best_iter=24 | val pids=['C8Q6']
  Fold 1: BA=0.3333 | best_iter=1 | val pids=['P4DZ']


/Users/dayana/git repo/machine-learning-class/lab3/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  Fold 2: BA=0.3694 | best_iter=13 | val pids=['F1ZM']


/Users/dayana/git repo/machine-learning-class/lab3/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:2924: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


  Fold 3: BA=0.5486 | best_iter=70 | val pids=['HDS9', 'TPQI']
  Fold 4: BA=0.3272 | best_iter=1 | val pids=['43JW', 'DT5C']

Mean per-fold BA: 0.4211 (std=0.0831)

=== Subject-disjoint OOF Balanced Accuracy: 0.3242 ===
Per-class recall (= diagonal of normalized CM):
  class 0: recall=0.1481  (n_true=162)
  class 1: recall=0.0000  (n_true=66)
  class 2: recall=0.8245  (n_true=587)

Confusion matrix (rows=true, cols=pred):
        pred_0  pred_1  pred_2
true_0      24       1     137
true_1       0       0      66
true_2      76      27     484


## 4. 테스트 예측

In [8]:
# 모든 (seed × fold) 모델의 평균 확률에서 argmax
test_proba_avg = test_proba_sum / n_models
test_pred = np.argmax(test_proba_avg, axis=1).astype(int)

print('Test prediction distribution:')
_cnt = pd.Series(test_pred).value_counts().sort_index()
_cnt.index = [f'class {i}' for i in _cnt.index]
print(_cnt)

# 모델이 단일 클래스로 붕괴되지 않았는지 sanity check
assert len(np.unique(test_pred)) >= 2, 'Predictions collapsed to a single class - investigate!'

Test prediction distribution:
class 0    488
class 1     61
class 2    479
Name: count, dtype: int64


# Generating Final Submissions


 Kaggle 제출물인 **csv 파일**을 생성하는 코드를 아래에 작성할 것

* 예.
```python
YOUR_PREDICTIONS.to_csv('submission-v1.csv')
```

In [9]:
# test_feats의 인덱스(=_id)는 TEST_LABEL의 id 순서를 그대로 보존하므로 이를 사용
submission = pd.DataFrame({
    'id': test_feats.index.astype(int),
    'stress': test_pred,
}).sort_values('id').reset_index(drop=True)

# 모든 test id가 포함되어 있는지 검증
_expected = set(TEST_LABEL.index.astype(int).tolist())
_got = set(submission['id'].tolist())
assert _expected == _got, f'Submission id mismatch: missing={_expected - _got}, extra={_got - _expected}'

submission.to_csv('submissioncc.csv', index=False)
print(f'Saved submission.csv: shape={submission.shape}')
print(submission.head())

Saved submission.csv: shape=(1028, 2)
    id  stress
0  815       1
1  816       1
2  817       2
3  818       2
4  819       2
